# Step 1: Generate datasets:

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import uuid
from datetime import datetime, timedelta

# Set seeds for reproducibility
Faker.seed(42)
fake = Faker()
np.random.seed(42)
random.seed(42)

NUM_CUSTOMERS = 2000
NUM_TRANSACTIONS = 30000

def generate_customers():
    customers = []
    country_codes = ['US', 'UK', 'IN', 'CA', 'AU']
    
    for _ in range(NUM_CUSTOMERS):
        # Intentional dirt: Mixed date formats
        date_format = random.choice(["%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y"])
        join_date = fake.date_between(start_date='-5y', end_date='today').strftime(date_format)
        
        # Intentional dirt: Inconsistent casing and whitespace in country codes
        country = random.choice(country_codes)
        if random.random() < 0.2:
            country = f" {country.lower()} "
            
        customers.append({
            'customer_id': str(uuid.uuid4()),
            'first_name': fake.first_name() if random.random() > 0.05 else None, # 5% missing names
            'last_name': fake.last_name(),
            'email': fake.email() if random.random() > 0.02 else np.nan, # 2% missing emails
            'join_date': join_date,
            'kyc_status': random.choice(['VERIFIED', 'PENDING', 'REJECTED', 'verified', 'Pending']), # Inconsistent casing
            'country_code': country
        })
        
    df_customers = pd.DataFrame(customers)
    
    # Intentional dirt: Duplicate customer records
    duplicates = df_customers.sample(n=50)
    df_customers = pd.concat([df_customers, duplicates], ignore_index=True)
    
    # Shuffle dataset
    df_customers = df_customers.sample(frac=1).reset_index(drop=True)
    return df_customers

def generate_transactions(valid_customer_ids):
    transactions = []
    
    for _ in range(NUM_TRANSACTIONS):
        # Intentional dirt: Orphan transactions (5% point to non-existent customers)
        if random.random() < 0.05:
            cid = str(uuid.uuid4()) 
        else:
            cid = random.choice(valid_customer_ids)
            
        # Intentional dirt: Negative amounts and extreme outliers
        amount = round(random.uniform(5.0, 5000.0), 2)
        if random.random() < 0.03:
            amount = amount * -1 
            
        # Intentional dirt: Mixed timestamp formats and timezones
        ts = fake.date_time_between(start_date='-5y', end_date='now')
        if random.random() < 0.5:
            ts_str = ts.isoformat() + "Z"
        else:
            ts_str = ts.strftime("%Y/%m/%d %H:%M:%S")
            
        transactions.append({
            'transaction_id': str(uuid.uuid4()),
            'customer_id': cid,
            'amount': amount,
            'currency': random.choice(['USD', 'EUR', 'GBP', 'usd', 'eur']), # Inconsistent casing
            'timestamp': ts_str,
            'transaction_type': random.choice(['credit', 'debit']),
            'status': random.choice(['COMPLETED', 'FAILED', 'PENDING', None]) # Null statuses
        })
        
    df_transactions = pd.DataFrame(transactions)
    
    # Intentional dirt: Duplicate transactions
    duplicates = df_transactions.sample(n=100)
    df_transactions = pd.concat([df_transactions, duplicates], ignore_index=True)
    
    return df_transactions

if __name__ == "__main__":
    print("Generating synthetic customers...")
    df_cust = generate_customers()
    df_cust.to_csv("customers_raw.csv", index=False)
    
    print("Generating synthetic transactions...")
    valid_cids = df_cust['customer_id'].dropna().tolist()
    df_trans = generate_transactions(valid_cids)
    df_trans.to_csv("transactions_raw.csv", index=False)
    
    print(f"Generated {len(df_cust)} customers and {len(df_trans)} transactions.")
    print("Files saved as 'customers_raw.csv' and 'transactions_raw.csv'.")

Generating synthetic customers...
Generating synthetic transactions...
Generated 2050 customers and 30100 transactions.
Files saved as 'customers_raw.csv' and 'transactions_raw.csv'.


# Step 2: Data Cleaning & Type Enforcement

In [2]:
import pandas as pd
# Read the CSV and convert them to DataFrames
customers_raw = pd.read_csv('customers_raw.csv')
transactions_raw = pd.read_csv('transactions_raw.csv')
# print(customers_raw.size)
# print(transactions_raw.size)

"""
********************************** PHASE 1: Deduplication **********************************************************
- The Problem: The generation script intentionally duplicated records in both tables. If left unhandled, this will inflate your transaction sums and distort customer counts.
- Your Task: Identify and remove duplicate rows across both datasets.
"""
# Clean the data, by deplucating it using the "df.drop_duplicates"
customers_cleaned = customers_raw.drop_duplicates(subset=["customer_id"])
transactions_cleaned = transactions_raw.drop_duplicates(subset=["transaction_id"])
# print(customers_cleaned.size)
# print(transactions_cleaned.size)

"""
********************************* PHASE 2: Text and Categorical Normalization **************************************
- The Problem: Categorical fields (country_code, kyc_status, currency) have inconsistent casing (e.g., " Pending", "PENDING") and hidden whitespace (" us ").
- Your Task: Standardize these columns so that grouping by them later works perfectly.
"""
# I will remove whitespaces (strip) and convert them to Uppercase (upper).
changes = ["country_code", "kyc_status"]
customers_cleaned[changes] = customers_cleaned[changes].apply(
    lambda x: x.str.strip().str.upper()
)

transactions_cleaned["currency"] = transactions_cleaned["currency"].str.strip().str.upper()

"""
********************************* PHASE 3: Temporal Standardization **************************************
- The Problem: join_date has mixed formats (YYYY-MM-DD, DD-MM-YYYY). The timestamp column in transactions has mixed formats and timezone representations (some ISO with Z, some without).
- Your Task: Convert both columns to native pandas datetime objects, ensuring the transaction timestamps are localized to UTC.
"""
from dateutil import parser
customers_cleaned["join_date"] = pd.to_datetime(customers_cleaned["join_date"].apply(parser.parse))
transactions_cleaned["timestamp"] = pd.to_datetime(transactions_cleaned["timestamp"].apply(parser.parse), utc=True)
transactions_cleaned.head()

"""
********************************* PHASE 4: Numeric Validation and Anomalies **************************************
- The Problem: The amount column contains negative values (e.g., -150.00), which violates the business rule that transaction amounts must be positive. 
- Your Task: Rectify the negative amounts.
"""
# You should treat the negative sign as a data error and standardize all amounts to be positive.
transactions_cleaned["amount"] = transactions_cleaned["amount"].abs()

"""
********************************* PHASE 5: Handling Missing Values (Nulls) **************************************
- The Problem: There are NaN or None values in first_name, email, and transaction status.
- Your Task: Decide how to handle these blanks explicitly.
"""
changes = ["first_name", "email"]
customers_cleaned[changes] = customers_cleaned[changes].apply(
    lambda x: x.fillna("NOT_PROVIDED")
)

transactions_cleaned = transactions_cleaned.dropna(subset=["status"])

# print(customers_cleaned.size)
# print(transactions_cleaned.size)


In [3]:
# My solution's rating was 8.5/10
# Solution:

import pandas as pd
# Read the CSV and convert them to DataFrames
customers_raw = pd.read_csv('customers_raw.csv')
transactions_raw = pd.read_csv('transactions_raw.csv')

"""
********************************** PHASE 1: Deduplication **********************************************************
- The Problem: The generation script intentionally duplicated records in both tables. If left unhandled, this will inflate your transaction sums and distort customer counts.
- Your Task: Identify and remove duplicate rows across both datasets.
"""
# Clean the data, by deplucating it using the "df.drop_duplicates"
customers_cleaned = customers_raw.drop_duplicates(subset=["customer_id"]).copy()
transactions_cleaned = transactions_raw.drop_duplicates(subset=["transaction_id"]).copy()

"""
********************************* PHASE 2: Text and Categorical Normalization **************************************
- The Problem: Categorical fields (country_code, kyc_status, currency) have inconsistent casing (e.g., " Pending", "PENDING") and hidden whitespace (" us ").
- Your Task: Standardize these columns so that grouping by them later works perfectly.
"""
# I will remove whitespaces (strip) and convert them to Uppercase (upper).
for col in ["country_code", "kyc_status"]:
    customers_cleaned[col] = customers_cleaned[col].str.strip().str.upper()

transactions_cleaned["currency"] = transactions_cleaned["currency"].str.strip().str.upper()

"""
********************************* PHASE 3: Temporal Standardization **************************************
- The Problem: join_date has mixed formats (YYYY-MM-DD, DD-MM-YYYY). The timestamp column in transactions has mixed formats and timezone representations (some ISO with Z, some without).
- Your Task: Convert both columns to native pandas datetime objects, ensuring the transaction timestamps are localized to UTC.
"""
customers_cleaned["join_date"] = pd.to_datetime(customers_cleaned["join_date"], format="mixed")
transactions_cleaned["timestamp"] = pd.to_datetime(transactions_cleaned["timestamp"], format="mixed", utc=True)

"""
********************************* PHASE 4: Numeric Validation and Anomalies **************************************
- The Problem: The amount column contains negative values (e.g., -150.00), which violates the business rule that transaction amounts must be positive. 
- Your Task: Rectify the negative amounts.
"""
# You should treat the negative sign as a data error and standardize all amounts to be positive.
transactions_cleaned["amount"] = transactions_cleaned["amount"].abs()

"""
********************************* PHASE 5: Handling Missing Values (Nulls) **************************************
- The Problem: There are NaN or None values in first_name, email, and transaction status.
- Your Task: Decide how to handle these blanks explicitly.
"""
customers_cleaned = customers_cleaned.fillna({"first_name" : "NOT_PROVIDED", "email": "NOT_PROVIDED"})
transactions_cleaned = transactions_cleaned.dropna(subset=["status"])

print(customers_cleaned)
print(transactions_cleaned)


                               customer_id    first_name  last_name  \
0     e7bbdaee-4959-46ac-a628-420ebc468788      Nicholas       Cobb   
1     fb2f8fe6-7aa2-410d-80e2-f5299eedd4e7   Christopher     Foster   
2     e87051d0-1f6e-4ccd-a1c6-d4215c79bf2f      Danielle      Smith   
3     7bed0faf-e74c-4630-9d92-448cebee0487          Lori      Flynn   
4     401d5db1-653d-4df3-b8c1-f1b46cbad4f7         Jorge     Murphy   
...                                    ...           ...        ...   
2044  91106979-654a-4c5d-bacb-2a0cb72358e4         Derek      Moore   
2045  24ed245f-4168-4798-8cc9-1c1cb1d5fc5e       Anthony  Zimmerman   
2046  82d266bc-c799-4df3-b86c-d510a2137e35  NOT_PROVIDED      Huang   
2048  aa838927-34c1-49a6-88b1-292988a88300        Angela      Jones   
2049  59067c1e-2f64-4f0d-83c3-d72e55c5b210       Zachary     Keller   

                             email  join_date kyc_status country_code  
0          morganjason@example.com 2023-01-05   REJECTED           AU  
1  

# Step 3: Relational Join & Feature Aggregation

In [16]:
"""
********************************** PHASE 1: Isolating the Orphans **********************************************************
- The Problem: The transaction data contains records pointing to customer_ids that do not exist in the customers_cleaned table. If we perform an inner join, we lose these transactions entirely. If we perform a left join from transactions to customers, we get transactions with null customer profiles.
- Your Task: Identify these orphans and filter them out of the main transaction dataset so they don't corrupt the aggregation, but optionally save them for auditing.
"""

# Extract Valid Customer Id's:
valid_customer_ids = customers_cleaned["customer_id"]

# Identify the orphans:
orphan_mask = ~transactions_cleaned["customer_id"].isin(valid_customer_ids)

# Store Orphan Transactions seprately:
orphan_transactions = transactions_cleaned[orphan_mask]

# Filter out Orphan Transactions:
transactions_cleaned = transactions_cleaned[
    transactions_cleaned["customer_id"].isin(valid_customer_ids)
] 

orphan_transactions.to_csv(
    "orphan_transactions.csv",
    index=False
)

"""
********************************** PHASE 2: Feature Aggregation **********************************************************
- The Problem: We need a single, flat table for the API to serve customer profiles efficiently. We cannot calculate total_spent, total_transactions, and last_active_date on the fly for every API request without massive latency.
- Your Task: Pre-compute these metrics.
"""

total_spent = (
    transactions_cleaned[
        transactions_cleaned["transaction_type"].eq("debit")
    ]
    .groupby("customer_id")["amount"]
    .sum()
)

grouped = transactions_cleaned.groupby("customer_id")

customer_activity = (
    transactions_cleaned
    .groupby("customer_id", as_index=False)
    .agg(
        total_transactions = ("transaction_id", "count"),
        last_active_date = ("timestamp", "max")
    )
)

"""
********************************** PHASE 3: The Merge (Left Join) **********************************************************
- The Problem: We need to combine the customer profiles with our newly calculated metrics.
- Your Task: Merge the dataframes on customer_id.
"""
customer_final = customers_cleaned.merge(
    customer_activity,
    on="customer_id",
    how="left"
)
customer_final["total_transactions"] = (
    customer_final["total_transactions"].fillna(0).astype(int)
)
customer_final.head()

,customer_id,first_name,last_name,email,join_date,kyc_status,country_code,total_transactions,last_active_date
0,e7bbdaee-4959-46ac-a628-420ebc468788,Nicholas,Cobb,morganjason@example.com,2023-01-05,REJECTED,AU,14,2026-08-12 12:25:01+00:00
1,fb2f8fe6-7aa2-410d-80e2-f5299eedd4e7,Christopher,Foster,amanda75@example.net,2024-12-15,PENDING,UK,10,2025-09-07 08:18:20+00:00
2,e87051d0-1f6e-4ccd-a1c6-d4215c79bf2f,Danielle,Smith,sharris@example.org,2026-09-14,REJECTED,CA,15,2026-05-13 15:10:36+00:00
3,7bed0faf-e74c-4630-9d92-448cebee0487,Lori,Flynn,ichambers@example.net,2022-07-06,VERIFIED,IN,9,2026-08-31 23:39:32+00:00
4,401d5db1-653d-4df3-b8c1-f1b46cbad4f7,Jorge,Murphy,wmurphy@example.com,2022-04-10,VERIFIED,CA,12,2025-12-06 11:46:17+00:00


In [23]:
# Solution:
"""
********************************** PHASE 1: Isolating the Orphans **********************************************************
- The Problem: The transaction data contains records pointing to customer_ids that do not exist in the customers_cleaned table. If we perform an inner join, we lose these transactions entirely. If we perform a left join from transactions to customers, we get transactions with null customer profiles.
- Your Task: Identify these orphans and filter them out of the main transaction dataset so they don't corrupt the aggregation, but optionally save them for auditing.
"""

# Extract Valid Customer Id's:
valid_customer_ids = customers_cleaned["customer_id"].unique()

is_valid = transactions_cleaned["customer_id"].isin(valid_customer_ids)

# Store Orphan Transactions seprately:
orphan_transactions = transactions_cleaned[~is_valid].copy()

orphan_transactions.to_csv(
    "orphan_transactions.csv",
    index=False
)

"""
********************************** PHASE 2: Feature Aggregation **********************************************************
- The Problem: We need a single, flat table for the API to serve customer profiles efficiently. We cannot calculate total_spent, total_transactions, and last_active_date on the fly for every API request without massive latency.
- Your Task: Pre-compute these metrics.
"""

total_spent = (
    transactions_cleaned[
        transactions_cleaned["transaction_type"].eq("debit")
    ]
    .groupby("customer_id")["amount"]
    .sum()
    .reset_index(name="total_spent")
)

customer_activity = (
    transactions_cleaned
    .groupby("customer_id", as_index=False)
    .agg(
        total_transactions = ("transaction_id", "count"),
        last_active_date = ("timestamp", "max")
    )
)

"""
********************************** PHASE 3: The Merge (Left Join) **********************************************************
- The Problem: We need to combine the customer profiles with our newly calculated metrics.
- Your Task: Merge the dataframes on customer_id.
"""
customer_final = customers_cleaned.merge(
    customer_activity,
    on="customer_id",
    how="left"
)

customer_final = customer_final.merge(
    total_spent,
    on="customer_id",
    how="left"
)

customer_final.fillna({
    "total_transactions": 0,
    "total_spent": 0.0
}, inplace=True)

customer_final["total_transactions"] = customer_final["total_transactions"].astype(int)
customer_final.head()

,customer_id,first_name,last_name,email,join_date,kyc_status,country_code,total_transactions,last_active_date,total_spent
0,e7bbdaee-4959-46ac-a628-420ebc468788,Nicholas,Cobb,morganjason@example.com,2023-01-05,REJECTED,AU,14,2026-08-12 12:25:01+00:00,11416.27
1,fb2f8fe6-7aa2-410d-80e2-f5299eedd4e7,Christopher,Foster,amanda75@example.net,2024-12-15,PENDING,UK,10,2025-09-07 08:18:20+00:00,25080.89
2,e87051d0-1f6e-4ccd-a1c6-d4215c79bf2f,Danielle,Smith,sharris@example.org,2026-09-14,REJECTED,CA,15,2026-05-13 15:10:36+00:00,17227.16
3,7bed0faf-e74c-4630-9d92-448cebee0487,Lori,Flynn,ichambers@example.net,2022-07-06,VERIFIED,IN,9,2026-08-31 23:39:32+00:00,5639.90
4,401d5db1-653d-4df3-b8c1-f1b46cbad4f7,Jorge,Murphy,wmurphy@example.com,2022-04-10,VERIFIED,CA,12,2025-12-06 11:46:17+00:00,17143.37


# Step 4: Storage & Persistence Layer

In [24]:
import sqlite3

conn = sqlite3.connect("bank_data.db")

customer_final.to_sql(
    name="customers",
    con=conn,
    if_exists="replace",
    index=False
)

transactions_cleaned.to_sql(
    name="transactions",
    con=conn,
    if_exists="replace",
    index=False
)

conn.close()

In [25]:
# Let's verify that both tables were successfully created.
conn = sqlite3.connect("bank_data.db")

cursor = conn.cursor()

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table';
""")

print(cursor.fetchall())

conn.close()

[('customers',), ('transactions',)]


# Step 5: FastAPI Service Architecture & Endpoints

In [ ]:
%pip install fastapi uvicorn pydantic

In [27]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bank_data.db")

print(pd.read_sql("PRAGMA table_info(customers)", conn))
print(pd.read_sql("PRAGMA table_info(transactions)", conn))

conn.close()

   cid                name       type  notnull dflt_value  pk
0    0         customer_id       TEXT        0       None   0
1    1          first_name       TEXT        0       None   0
2    2           last_name       TEXT        0       None   0
3    3               email       TEXT        0       None   0
4    4           join_date  TIMESTAMP        0       None   0
5    5          kyc_status       TEXT        0       None   0
6    6        country_code       TEXT        0       None   0
7    7  total_transactions    INTEGER        0       None   0
8    8    last_active_date  TIMESTAMP        0       None   0
9    9         total_spent       REAL        0       None   0
   cid              name       type  notnull dflt_value  pk
0    0    transaction_id       TEXT        0       None   0
1    1       customer_id       TEXT        0       None   0
2    2            amount       REAL        0       None   0
3    3          currency       TEXT        0       None   0
4    4         tim

In [ ]:
# Inside schemas.py

# Step 6: Testing & Quality Assurance